# Notebook 1: LCA Database Cleanup

**Project:** RAW Project — 3D-Printed Biopolymer Panels  
**Database:** `lca_database_3DPrintedBiopol`  
**Ecoinvent version:** 3.12, cut-off system model  
**Date:** 2026  

## Purpose

This notebook cleans up the existing foreground LCA database. It makes no new modelling decisions — it only corrects structural errors and consolidates activities that were created as placeholders.

### Changes made in this notebook

1. **Fix the process chain sequence**: the drying → kiln baking sequence was inverted. Corrected order: `3D printing → Drying → Kiln baking`.
2. **Consolidate filler activities**: four fillers (bark, cellulose, cotton, wood flour) each had a standalone milling activity disconnected from the corresponding filler production activity. These are merged into single consolidated activities per filler.

### What is NOT changed in this notebook

- Hemp dust and seagrass filler production (proxy datasets to be reviewed separately)
- Pea protein binder production (to be rebuilt from scratch in Notebook 2)
- The avoided burden activity for nitrogen fixation
- All exchange amounts in the 3D printing activity (zero values are intentional placeholders for parameterisation)

---

## 0. Setup and sanity check

In [1]:
import bw2data as bd
import bw2io as bi

# Set the active project
bd.projects.set_current("biopol_lca")

# Open the foreground database
db = bd.Database("lca_database_3DPrintedBiopol")

# Open the ecoinvent background database
ei = bd.Database("ecoinvent-3.12-cutoff")

print(f"Foreground database: {len(db)} activities")
print()
print("Activities currently in database:")
for act in sorted(db, key=lambda x: x['name']):
    exchanges = list(act.exchanges())
    inputs = [e for e in exchanges if e['type'] == 'technosphere']
    print(f"  {act['name']} ({act['location']}) — {len(inputs)} technosphere input(s)")

Foreground database: 14 activities

Activities currently in database:
  3D printing (RER) — 11 technosphere input(s)
  Drying with dehumidifying chamber (RER) — 3 technosphere input(s)
  Kiln baking - 150 degress for 2 hours (RER) — 3 technosphere input(s)
  avoided burden: nitrogen fixation, peas (GLO) — 1 technosphere input(s)
  bark chip milling (GLO) — 3 technosphere input(s)
  bark flour filler production (GLO) — 1 technosphere input(s)
  cellulose fiber milling (GLO) — 3 technosphere input(s)
  cellulose filler production (GLO) — 1 technosphere input(s)
  cotton filler production (GLO) — 1 technosphere input(s)
  hemp dust filler production (GLO) — 2 technosphere input(s)
  pea protein binder production (GLO) — 1 technosphere input(s)
  seagrass filler production (GLO) — 2 technosphere input(s)
  seed-cotton milling (GLO) — 3 technosphere input(s)
  wood flour filler production (GLO) — 2 technosphere input(s)


---

## 1. Fix the process chain sequence

### Background

The 3D-printed biopolymer panel passes through three sequential manufacturing steps:

```
3D printing  →  Drying with dehumidifying chamber  →  Kiln baking (150°C, 2h)
```

In the original database, this chain was inverted: `Drying` consumed `Kiln baking` as an input, and `Kiln baking` consumed `3D printing`. This cell corrects the linkages so each activity consumes the output of the previous step.

### Helper function

In [2]:
def find_activity(db, name):
    """Find a single activity by exact name. Raises an error if not found or ambiguous."""
    results = [a for a in db if a['name'] == name]
    if len(results) == 0:
        raise ValueError(f"Activity not found: '{name}'")
    if len(results) > 1:
        raise ValueError(f"Multiple activities found for: '{name}'")
    return results[0]


def delete_exchange(activity, input_activity):
    """Delete all technosphere exchanges linking to input_activity from activity."""
    to_delete = [
        exc for exc in activity.exchanges()
        if exc['type'] == 'technosphere'
        and exc.input.key == input_activity.key
    ]
    for exc in to_delete:
        exc.delete()
    return len(to_delete)

In [3]:
# Retrieve the three process chain activities
act_printing = find_activity(db, "3D printing")
act_drying   = find_activity(db, "Drying with dehumidifying chamber")
act_kiln     = find_activity(db, "Kiln baking - 150 degress for 2 hours")

print("Activities retrieved:")
print(f"  3D printing         → reference product: {act_printing['reference product']}")
print(f"  Drying              → reference product: {act_drying['reference product']}")
print(f"  Kiln baking         → reference product: {act_kiln['reference product']}")

Activities retrieved:
  3D printing         → reference product: Biopolymer panel, 3D printed
  Drying              → reference product: Biopolymer panel, dried
  Kiln baking         → reference product: Biopolymer panel, baked


In [4]:
# --- Fix Drying ---
# Remove the incorrect input (kiln baking) and replace with 3D printing output

n_deleted = delete_exchange(act_drying, act_kiln)
print(f"Drying: removed {n_deleted} exchange(s) pointing to kiln baking")

act_drying.new_exchange(
    input=act_printing,
    amount=1.0,
    type='technosphere',
    comment="Output of 3D printing step; dried in dehumidifying chamber before kiln baking"
).save()
act_drying.save()
print("Drying: added input → '3D printing' (Biopolymer panel, 3D printed), 1.0 kg")

Drying: removed 1 exchange(s) pointing to kiln baking
Drying: added input → '3D printing' (Biopolymer panel, 3D printed), 1.0 kg


In [5]:
# --- Fix Kiln baking ---
# Remove the incorrect input (3D printing) and replace with drying output

n_deleted = delete_exchange(act_kiln, act_printing)
print(f"Kiln baking: removed {n_deleted} exchange(s) pointing to 3D printing")

act_kiln.new_exchange(
    input=act_drying,
    amount=1.0,
    type='technosphere',
    comment="Output of drying step; baked at 150°C for 2 hours per Eleftheria protocol"
).save()
act_kiln.save()
print("Kiln baking: added input → 'Drying with dehumidifying chamber' (Biopolymer panel, dried), 1.0 kg")

Kiln baking: removed 1 exchange(s) pointing to 3D printing
Kiln baking: added input → 'Drying with dehumidifying chamber' (Biopolymer panel, dried), 1.0 kg


In [6]:
# Verify the corrected chain
print("=== Corrected process chain ===")
print()

for act in [act_printing, act_drying, act_kiln]:
    print(f"Activity: {act['name']}")
    print(f"  Output: {act['reference product']}")
    tech_inputs = [e for e in act.exchanges() if e['type'] == 'technosphere']
    for e in tech_inputs:
        print(f"  Input:  {e.input['name']} | {e.input['reference product']} | {e['amount']} {act['unit']}")
    print()

=== Corrected process chain ===

Activity: 3D printing
  Output: Biopolymer panel, 3D printed
  Input:  market for electricity, medium voltage | electricity, medium voltage | 0.11764705882352942 kilogram
  Input:  industrial machine production, heavy, unspecified | industrial machine, heavy, unspecified | 0.0013430029546065002 kilogram
  Input:  hemp dust filler production | hemp dust filler | 0.22 kilogram
  Input:  pea protein binder production | pea protein binder | 0.72 kilogram
  Input:  seagrass filler production | seagrass filler | 0.06 kilogram
  Input:  market for glycerine | glycerine | 0.0 kilogram
  Input:  market for calcium chloride | calcium chloride | 0.0 kilogram
  Input:  bark flour filler production | bark flour filler | 0.0 kilogram
  Input:  wood flour filler production | wood flour filler | 0.0 kilogram
  Input:  cotton filler production | cotton filler | 0.0 kilogram
  Input:  cellulose filler production | cellulose filler | 0.0 kilogram

Activity: Drying with de

---

## 2. Consolidate filler activities

### Background

Four fillers — bark, cellulose, cotton, and wood flour — each had two disconnected activities:

- A **milling activity** containing the raw material input, electricity for milling, and machine wear
- A **filler production activity** containing only transport

The milling activities were never linked to the filler production activities, so they contributed nothing to the LCA results. This section merges each pair into a single consolidated filler production activity with all inputs present, then deletes the now-redundant milling activities.

### Electricity sources

All milling electricity values are sourced from:

> Wang, J., Gao, J., Brandt, K.L. et al. (2018). *Energy consumption of two-stage fine grinding of Douglas-fir wood.* Journal of Wood Science, 64, 338–346. https://doi.org/10.1007/s10086-018-1712-1

This paper investigated two-stage hammer mill + rotor impact mill grinding of wood feedstocks to fine powders. Values vary by material and target particle size. All fillers in the RAW project are milled to 0.25 mm (250 µm) using the Retsch SM2000 cutting mill (per Eleftheria's protocol).

Cotton electricity is sourced from:

> Funk, P.A. et al. (2012–2013). *Cotton Ginning Handbook: Energy Utilization and Conservation in Cotton Gins.* USDA. Available: https://www.researchgate.net/publication/365020363

The handbook reports average electricity use of ~35 kWh per bale (~227 kg), with a range of 150–275 kWh/tonne. The value of 0.212 kWh/kg used here sits within that range.

### Machine wear methodology

Machine wear is modelled using the ecoinvent dataset `market for industrial machine, heavy, unspecified | RER` as a proxy for the Retsch SM2000 cutting mill.

**Amount derivation (lab-scale):**
- Machine mass: ~150 kg (laboratory cutting mill)
- Lifespan: 15 years (mid-point of 10–20 year range for well-maintained industrial mills)
- Operating hours: ~1,000 h/year (conservative for lab use)
- Throughput: 1 kg/h (per Eleftheria protocol: 5 kg per 5 hours)
- Lifetime throughput: 15 × 1,000 × 1 = 15,000 kg
- **Wear per kg output: 150 / 15,000 = 0.01 kg machine per kg filler**

> ⚠️ **Prospective LCA note:** This machine wear value reflects the lab-scale setup used in the current experimental protocol (DTU/CITA). In a prospective LCA representing industrial-scale production, throughput would be orders of magnitude higher (industrial hammer mills process 1–10 tonnes/hour), and machine wear per kg of output would be substantially lower — likely in the range of 1×10⁻⁴ to 1×10⁻⁶ kg/kg. This should be revised when scaling up the model.

### Transport

Transport is modelled as `market for transport, freight, lorry, unspecified | RER`, assuming a transport distance of **50 km** for 1 kg of material:

```
50 km × 0.001 tonne = 0.05 tonne-km (tkm)
```

### Parameters

In [7]:
# ── Shared parameters ──────────────────────────────────────────────────────────

MACHINE_WEAR_KG_PER_KG = 0.01
# Lab-scale estimate: 150 kg machine / 15,000 kg lifetime throughput.
# See methodology note above. Revise for industrial-scale prospective LCA.

TRANSPORT_TKM = 0.05
# 50 km × 0.001 tonne = 0.05 tkm per kg of material

# ── Electricity values by filler (kWh per kg output) ──────────────────────────

ELECTRICITY = {
    "bark":      {"kwh": 0.35,  "source": "Wang et al. 2018, J Wood Sci 64:338-346, doi:10.1007/s10086-018-1712-1"},
    "cellulose": {"kwh": 0.67,  "source": "Wang et al. 2018, J Wood Sci 64:338-346, doi:10.1007/s10086-018-1712-1. Higher than bark due to finer particle requirements for cellulose fibre."},
    "cotton":    {"kwh": 0.212, "source": "Cotton Ginning Handbook (USDA, Funk et al.), ResearchGate ID 365020363. Range 150-275 kWh/tonne; 0.212 kWh/kg used as mid-to-upper estimate."},
    "woodflour": {"kwh": 0.15,  "source": "Wang et al. 2018, J Wood Sci 64:338-346, doi:10.1007/s10086-018-1712-1. Estimated for 250 µm target particle size (first grinding stage). Literature range 0.1-0.2 kWh/kg at this size."}
}

print("Parameters set:")
print(f"  Machine wear:  {MACHINE_WEAR_KG_PER_KG} kg machine / kg output")
print(f"  Transport:     {TRANSPORT_TKM} tkm / kg")
print()
print("Electricity by filler:")
for filler, vals in ELECTRICITY.items():
    print(f"  {filler:<12} {vals['kwh']} kWh/kg")

Parameters set:
  Machine wear:  0.01 kg machine / kg output
  Transport:     0.05 tkm / kg

Electricity by filler:
  bark         0.35 kWh/kg
  cellulose    0.67 kWh/kg
  cotton       0.212 kWh/kg
  woodflour    0.15 kWh/kg


In [8]:
# ── Retrieve shared background datasets ───────────────────────────────────────

def find_ei(name, location, ref_product=None):
    """Look up an ecoinvent activity by name, location, and optionally reference product."""
    results = [
        a for a in ei
        if a['name'] == name and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Ecoinvent dataset not found: '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} — using first")
    return results[0]


ei_electricity = find_ei(
    "market for electricity, medium voltage", "DE",
    ref_product="electricity, medium voltage"
)
ei_machine = find_ei(
    "market for industrial machine, heavy, unspecified", "RER",
    ref_product="industrial machine, heavy, unspecified"
)
ei_transport = find_ei(
    "market for transport, freight, lorry, unspecified", "RER",
    ref_product="transport, freight, lorry, diesel, unspecified"
)

print("Background datasets retrieved:")
print(f"  Electricity: {ei_electricity['name']} | {ei_electricity['location']}")
print(f"  Machine:     {ei_machine['name']} | {ei_machine['location']}")
print(f"  Transport:   {ei_transport['name']} | {ei_transport['location']}")

Background datasets retrieved:
  Electricity: market for electricity, medium voltage | DE
  Machine:     market for industrial machine, heavy, unspecified | RER
  Transport:   market for transport, freight, lorry, unspecified | RER


### 2.1 Bark filler

**Raw material:** `market for bark chips, green, measured as dry mass | Europe without Switzerland`  
**Electricity:** 0.35 kWh/kg — Wang et al. 2018  
**Milling activity to delete:** `bark chip milling`

In [9]:
# Retrieve activities
act_bark_filler  = find_activity(db, "bark flour filler production")
act_bark_milling = find_activity(db, "bark chip milling")

# Raw material dataset
ei_bark_raw = find_ei(
    "market for bark chips, green, measured as dry mass",
    "Europe without Switzerland",
    ref_product="bark chips, green, measured as dry mass"
)

# ── Update transport to correct tkm amount ─────────────────────────────────────
for exc in act_bark_filler.exchanges():
    if exc['type'] == 'technosphere' and 'transport' in exc.input['name'].lower():
        exc['amount'] = TRANSPORT_TKM
        exc['comment'] = "50 km × 0.001 tonne = 0.05 tkm per kg of bark chips"
        exc.save()

# ── Add raw material ───────────────────────────────────────────────────────────
act_bark_filler.new_exchange(
    input=ei_bark_raw,
    amount=1.0,
    type='technosphere',
    comment="Bark chips, green, dry mass basis. Source: ecoinvent 3.12 cut-off, Europe without Switzerland."
).save()

# ── Add electricity ────────────────────────────────────────────────────────────
act_bark_filler.new_exchange(
    input=ei_electricity,
    amount=ELECTRICITY["bark"]["kwh"],
    type='technosphere',
    comment=ELECTRICITY["bark"]["source"]
).save()

# ── Add machine wear ───────────────────────────────────────────────────────────
act_bark_filler.new_exchange(
    input=ei_machine,
    amount=MACHINE_WEAR_KG_PER_KG,
    type='technosphere',
    comment=(
        f"Machine wear: {MACHINE_WEAR_KG_PER_KG} kg machine per kg output. "
        "Lab-scale estimate based on Retsch SM2000 cutting mill (~150 kg, 15-year lifespan, "
        "1,000 h/year, 1 kg/h throughput per Eleftheria protocol). "
        "For industrial-scale prospective LCA, revise to ~1e-4 to 1e-6 kg/kg."
    )
).save()

act_bark_filler.save()

# ── Delete the standalone milling activity ─────────────────────────────────────
act_bark_milling.delete()

print("Bark filler production — consolidated. Bark chip milling deleted.")

Bark filler production — consolidated. Bark chip milling deleted.


### 2.2 Cellulose filler

**Raw material:** `market for cellulose fibre | RoW`  
**Electricity:** 0.67 kWh/kg — Wang et al. 2018 (higher than bark; cellulose fibre requires more energy at 250 µm target size)  
**Milling activity to delete:** `cellulose fiber milling`

In [10]:
act_cellulose_filler  = find_activity(db, "cellulose filler production")
act_cellulose_milling = find_activity(db, "cellulose fiber milling")

ei_cellulose_raw = find_ei(
    "market for cellulose fibre", "RoW",
    ref_product="cellulose fibre"
)

# Update transport
for exc in act_cellulose_filler.exchanges():
    if exc['type'] == 'technosphere' and 'transport' in exc.input['name'].lower():
        exc['amount'] = TRANSPORT_TKM
        exc['comment'] = "50 km × 0.001 tonne = 0.05 tkm per kg of cellulose fibre"
        exc.save()

# Add raw material
act_cellulose_filler.new_exchange(
    input=ei_cellulose_raw,
    amount=1.0,
    type='technosphere',
    comment="Cellulose fibre, RoW. Source: ecoinvent 3.12 cut-off."
).save()

# Add electricity
act_cellulose_filler.new_exchange(
    input=ei_electricity,
    amount=ELECTRICITY["cellulose"]["kwh"],
    type='technosphere',
    comment=ELECTRICITY["cellulose"]["source"]
).save()

# Add machine wear
act_cellulose_filler.new_exchange(
    input=ei_machine,
    amount=MACHINE_WEAR_KG_PER_KG,
    type='technosphere',
    comment=(
        f"Machine wear: {MACHINE_WEAR_KG_PER_KG} kg machine per kg output. "
        "Lab-scale estimate based on Retsch SM2000 cutting mill (~150 kg, 15-year lifespan, "
        "1,000 h/year, 1 kg/h throughput per Eleftheria protocol). "
        "For industrial-scale prospective LCA, revise to ~1e-4 to 1e-6 kg/kg."
    )
).save()

act_cellulose_filler.save()
act_cellulose_milling.delete()

print("Cellulose filler production — consolidated. Cellulose fiber milling deleted.")

Cellulose filler production — consolidated. Cellulose fiber milling deleted.


### 2.3 Cotton filler

**Raw material:** `market for seed-cotton | GLO`  
**Electricity:** 0.212 kWh/kg — Cotton Ginning Handbook (USDA, Funk et al.)  
**Milling activity to delete:** `seed-cotton milling`

In [11]:
act_cotton_filler  = find_activity(db, "cotton filler production")
act_cotton_milling = find_activity(db, "seed-cotton milling")

ei_cotton_raw = find_ei(
    "market for seed-cotton", "GLO",
    ref_product="seed-cotton"
)

# Update transport
for exc in act_cotton_filler.exchanges():
    if exc['type'] == 'technosphere' and 'transport' in exc.input['name'].lower():
        exc['amount'] = TRANSPORT_TKM
        exc['comment'] = "50 km × 0.001 tonne = 0.05 tkm per kg of seed-cotton"
        exc.save()

# Add raw material
act_cotton_filler.new_exchange(
    input=ei_cotton_raw,
    amount=1.0,
    type='technosphere',
    comment="Seed-cotton, GLO. Source: ecoinvent 3.12 cut-off."
).save()

# Add electricity
act_cotton_filler.new_exchange(
    input=ei_electricity,
    amount=ELECTRICITY["cotton"]["kwh"],
    type='technosphere',
    comment=ELECTRICITY["cotton"]["source"]
).save()

# Add machine wear
act_cotton_filler.new_exchange(
    input=ei_machine,
    amount=MACHINE_WEAR_KG_PER_KG,
    type='technosphere',
    comment=(
        f"Machine wear: {MACHINE_WEAR_KG_PER_KG} kg machine per kg output. "
        "Lab-scale estimate based on Retsch SM2000 cutting mill (~150 kg, 15-year lifespan, "
        "1,000 h/year, 1 kg/h throughput per Eleftheria protocol). "
        "For industrial-scale prospective LCA, revise to ~1e-4 to 1e-6 kg/kg."
    )
).save()

act_cotton_filler.save()
act_cotton_milling.delete()

print("Cotton filler production — consolidated. Seed-cotton milling deleted.")

Cotton filler production — consolidated. Seed-cotton milling deleted.


### 2.4 Wood flour filler

**Raw material:** `market for sawdust, green, collected, measured as dry mass | Europe without Switzerland`  
(already present in the original activity)

**Electricity:** 0.15 kWh/kg — Wang et al. 2018, estimated for 250 µm target particle size.  
Literature range at this particle size is 0.1–0.2 kWh/kg; 0.15 kWh/kg used as central estimate.

**Note:** No standalone milling activity existed for wood flour — this filler only needs electricity, machine wear, and transport corrections applied to the existing activity.

In [12]:
act_woodflour_filler = find_activity(db, "wood flour filler production")

# Update transport (raw material sawdust already present — no need to re-add)
for exc in act_woodflour_filler.exchanges():
    if exc['type'] == 'technosphere' and 'transport' in exc.input['name'].lower():
        exc['amount'] = TRANSPORT_TKM
        exc['comment'] = "50 km × 0.001 tonne = 0.05 tkm per kg of sawdust"
        exc.save()

# Add electricity
act_woodflour_filler.new_exchange(
    input=ei_electricity,
    amount=ELECTRICITY["woodflour"]["kwh"],
    type='technosphere',
    comment=ELECTRICITY["woodflour"]["source"]
).save()

# Add machine wear
act_woodflour_filler.new_exchange(
    input=ei_machine,
    amount=MACHINE_WEAR_KG_PER_KG,
    type='technosphere',
    comment=(
        f"Machine wear: {MACHINE_WEAR_KG_PER_KG} kg machine per kg output. "
        "Lab-scale estimate based on Retsch SM2000 cutting mill (~150 kg, 15-year lifespan, "
        "1,000 h/year, 1 kg/h throughput per Eleftheria protocol). "
        "For industrial-scale prospective LCA, revise to ~1e-4 to 1e-6 kg/kg."
    )
).save()

act_woodflour_filler.save()

print("Wood flour filler production — electricity and machine wear added.")
print("(No standalone milling activity existed for wood flour — nothing to delete.)")

Wood flour filler production — electricity and machine wear added.
(No standalone milling activity existed for wood flour — nothing to delete.)


---

## 3. Verification

Print the final state of all modified activities and confirm the database is in the expected state.

In [13]:
# ── 3a. Verify process chain ───────────────────────────────────────────────────
print("=" * 60)
print("PROCESS CHAIN VERIFICATION")
print("=" * 60)

for act in [act_printing, act_drying, act_kiln]:
    print(f"\n▶ {act['name']}")
    print(f"  Reference product: {act['reference product']}")
    for exc in act.exchanges():
        if exc['type'] == 'technosphere':
            print(f"  INPUT  {exc['amount']:>10.6f} {exc.input['unit']:<12} {exc.input['name']}")
        elif exc['type'] == 'production':
            print(f"  OUTPUT {exc['amount']:>10.6f} {exc.input['unit']:<12} {exc.input['reference product']}")

PROCESS CHAIN VERIFICATION

▶ 3D printing
  Reference product: Biopolymer panel, 3D printed
  OUTPUT   1.000000 kilogram     Biopolymer panel, 3D printed
  INPUT    0.117647 kilowatt hour market for electricity, medium voltage
  INPUT    0.001343 kilogram     industrial machine production, heavy, unspecified
  INPUT    0.220000 kilogram     hemp dust filler production
  INPUT    0.720000 kilogram     pea protein binder production
  INPUT    0.060000 kilogram     seagrass filler production
  INPUT    0.000000 kilogram     market for glycerine
  INPUT    0.000000 kilogram     market for calcium chloride
  INPUT    0.000000 kilogram     bark flour filler production
  INPUT    0.000000 kilogram     wood flour filler production
  INPUT    0.000000 kilogram     cotton filler production
  INPUT    0.000000 kilogram     cellulose filler production

▶ Drying with dehumidifying chamber
  Reference product: Biopolymer panel, dried
  OUTPUT   1.000000 kilogram     Biopolymer panel, dried
  INPUT  

In [14]:
# ── 3b. Verify consolidated filler activities ─────────────────────────────────
print("=" * 60)
print("FILLER ACTIVITIES VERIFICATION")
print("=" * 60)

filler_activities = [
    act_bark_filler,
    act_cellulose_filler,
    act_cotton_filler,
    act_woodflour_filler,
]

for act in filler_activities:
    print(f"\n▶ {act['name']}")
    print(f"  Reference product: {act['reference product']}")
    for exc in act.exchanges():
        if exc['type'] == 'technosphere':
            print(f"  INPUT  {exc['amount']:>12.6f} {exc.input['unit']:<15} {exc.input['name']} | {exc.input.get('location', '')}")
        elif exc['type'] == 'production':
            print(f"  OUTPUT {exc['amount']:>12.6f} {exc.input['unit']:<15} {exc.input['reference product']}")

FILLER ACTIVITIES VERIFICATION

▶ bark flour filler production
  Reference product: bark flour filler
  OUTPUT     1.000000 kilogram        bark flour filler
  INPUT      0.050000 ton kilometer   market for transport, freight, lorry, unspecified | RER
  INPUT      1.000000 kilogram        market for bark chips, green, measured as dry mass | Europe without Switzerland
  INPUT      0.350000 kilowatt hour   market for electricity, medium voltage | DE
  INPUT      0.010000 kilogram        market for industrial machine, heavy, unspecified | RER

▶ cellulose filler production
  Reference product: cellulose filler
  OUTPUT     1.000000 kilogram        cellulose filler
  INPUT      0.050000 ton kilometer   market for transport, freight, lorry, unspecified | RER
  INPUT      1.000000 kilogram        market for cellulose fibre | RoW
  INPUT      0.670000 kilowatt hour   market for electricity, medium voltage | DE
  INPUT      0.010000 kilogram        market for industrial machine, heavy, unspeci

In [15]:
# ── 3c. Confirm standalone milling activities are deleted ─────────────────────
print("=" * 60)
print("DELETED MILLING ACTIVITIES CHECK")
print("=" * 60)

milling_names = [
    "bark chip milling",
    "cellulose fiber milling",
    "seed-cotton milling",
]

# Refresh the database view
db = bd.Database("lca_database_3DPrintedBiopol")
current_names = {a['name'] for a in db}

for name in milling_names:
    status = "❌ STILL EXISTS — check deletion" if name in current_names else "✓ deleted"
    print(f"  {name}: {status}")

print()
print("Final database activity list:")
for act in sorted(db, key=lambda x: x['name']):
    print(f"  {act['name']} ({act['location']})")

DELETED MILLING ACTIVITIES CHECK
  bark chip milling: ✓ deleted
  cellulose fiber milling: ✓ deleted
  seed-cotton milling: ✓ deleted

Final database activity list:
  3D printing (RER)
  Drying with dehumidifying chamber (RER)
  Kiln baking - 150 degress for 2 hours (RER)
  avoided burden: nitrogen fixation, peas (GLO)
  bark flour filler production (GLO)
  cellulose filler production (GLO)
  cotton filler production (GLO)
  hemp dust filler production (GLO)
  pea protein binder production (GLO)
  seagrass filler production (GLO)
  wood flour filler production (GLO)


---

## Summary of changes

| Change | Status |
|--------|--------|
| Process chain: Drying now takes 3D printing output as input | ✓ |
| Process chain: Kiln baking now takes Drying output as input | ✓ |
| Bark filler: raw material + electricity + machine wear + transport added | ✓ |
| Cellulose filler: raw material + electricity + machine wear + transport added | ✓ |
| Cotton filler: raw material + electricity + machine wear + transport added | ✓ |
| Wood flour filler: electricity + machine wear + transport added | ✓ |
| Bark chip milling activity deleted | ✓ |
| Cellulose fiber milling activity deleted | ✓ |
| Seed-cotton milling activity deleted | ✓ |
| Hemp dust, seagrass, pea protein — unchanged | ✓ |
| 3D printing zero-value exchanges — unchanged | ✓ |

**Next steps:**
- Notebook 2: Rebuild pea protein chain from scratch (pea cultivation → AEIEP isolate production → binder production) with correct mass balances and nitrogen fixation avoided burden
- Notebook 3: Seagrass and cellulose avoided burden accounting